# NYC Yellow Taxi Fare Prediction — Exploratory Data Analysis (EDA)
### Enterprise ML Pipeline POC: Spark + Airflow + scikit-learn + PyTorch + MLflow

This notebook demonstrates preliminary exploration of the NYC Yellow Taxi Parquet trip records,
analyzing fare distributions, trip distance correlations, temporal patterns, and pickup zones.

In [ ]:
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Libraries loaded successfully!")

## 1. Load Sample Parquet Data

In [ ]:
parquet_files = sorted(glob.glob("../Data_Files/*.parquet") + glob.glob("../Data files/*.parquet") + glob.glob("Data_Files/*.parquet"))
print(f"Found {len(parquet_files)} Parquet files:")
for f in parquet_files:
    print(f"  - {f} ({os.path.getsize(f) / 1024:.1f} KB)")

# Read the first available monthly file
if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    print(f"\nLoaded {len(df):,} records from {parquet_files[0]}")
    display(df.head())
else:
    print("No parquet files found. Run 'python scripts/generate_sample_data.py' first!")

## 2. Statistical Summary of Key Features

In [ ]:
cols = ["trip_distance", "passenger_count", "fare_amount", "total_amount"]
if 'df' in locals():
    display(df[cols].describe().round(2))

## 3. Visualizing Fare Amount vs Trip Distance

In [ ]:
if 'df' in locals():
    # Filter outliers for clean visualization
    clean_sample = df[(df["fare_amount"] > 2.5) & (df["fare_amount"] < 150) & (df["trip_distance"] > 0) & (df["trip_distance"] < 30)]
    
    plt.figure(figsize=(10, 5))
    plt.scatter(clean_sample["trip_distance"], clean_sample["fare_amount"], alpha=0.3, color="#2b5c8f", s=12)
    plt.title("NYC Yellow Taxi: Trip Distance vs Fare Amount ($)", fontsize=13, fontweight="bold")
    plt.xlabel("Trip Distance (miles)", fontsize=11)
    plt.ylabel("Fare Amount ($)", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

## 4. Hourly Demand and Fare Dynamics

In [ ]:
if 'df' in locals():
    df["pickup_hour"] = pd.to_datetime(df["tpep_pickup_datetime"]).dt.hour
    hourly_stats = df.groupby("pickup_hour")["fare_amount"].agg(["count", "mean"])
    
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()
    
    ax1.bar(hourly_stats.index, hourly_stats["count"], alpha=0.6, color="skyblue", label="Trip Volume")
    ax2.plot(hourly_stats.index, hourly_stats["mean"], color="crimson", lw=2, marker="o", label="Avg Fare ($)")
    
    ax1.set_xlabel("Pickup Hour of Day (0-23)", fontsize=11)
    ax1.set_ylabel("Total Trip Volume", color="navy", fontsize=11)
    ax2.set_ylabel("Average Fare ($)", color="crimson", fontsize=11)
    plt.title("Hourly Trip Volume and Average Fare Pattern", fontsize=13, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.show()